# Tarea 2.2 · Proof of Concept funcional en PySpark

**Curso:** Herramientas de Big Data · Saber 2: Procesamiento distribuido
**Proyecto:** Emergencias de la Línea 123 de Bogotá (julio, agosto y septiembre de 2026)
**Actividad:** Actividad 2 · Diseño de arquitectura y PoC (evaluativa)

**Objetivos**
- Configurar PySpark en Colab e ingerir los 3 CSV directamente desde Google Drive.
- Limpiar nulos, convertir tipos y estandarizar códigos, sin borrar ni imputar registros (solo se apartan los duplicados exactos).
- Validar la calidad (conteos, rangos de fechas, consistencia de códigos), exportar a Parquet en Drive y dejar un log de validación en .txt.

**Prerrequisitos**
- Los 3 CSV en `MyDrive/BigData_U2/datos/` y la tabla de códigos que deja el notebook 01.
- Notebooks del curso 02 (SparkSession), 03 (explain, nulos, outliers, broadcast) y 04 (pipeline robusto).

**Tiempo estimado de ejecución:** ~3 minutos (Colab gratuito, solo CPU)

**Dataset:** «Llamadas de urgencias y emergencias que ingresan a través de la línea 123», Secretaría Distrital de
Salud, portal Datos Abiertos Bogotá (CC BY 4.0). Tres CSV mensuales (27 434 registros, separador `;`).

## 1. Configuración del entorno

In [ ]:
# 1.1 Conexión con Google Drive (los datos y los resultados viven en MyDrive/BigData_U2)
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 1.2 Librerías
import time, json, inspect, threading, platform, gc, shutil
import numpy as np
import pandas as pd
import polars as pl
import psutil
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

os.environ['TZ'] = 'America/Bogota'   # hora local en los logs
time.tzset()

# 1.3 Rutas de trabajo (una sola carpeta del proyecto en Drive)
BASE = '/content/drive/MyDrive/BigData_U2'
DATOS = f'{BASE}/datos'
RESULTADOS = f'{BASE}/resultados'
MEDICIONES = f'{RESULTADOS}/mediciones'
for ruta in (RESULTADOS, MEDICIONES, f'{RESULTADOS}/logs', f'{RESULTADOS}/parquet'):
    os.makedirs(ruta, exist_ok=True)

ARCHIVOS = ['llamadas123_julio_2026.csv', 'llamadaslinea123agosto2026.csv', 'llamadaslinea123Septiembre.csv']
RUTAS = [f'{DATOS}/{a}' for a in ARCHIVOS]
COLUMNAS = ['NUMERO_INCIDENTE', 'FECHA_INICIO_DESPLAZAMIENTO_MOVIL', 'CODIGO_LOCALIDAD', 'LOCALIDAD', 'EDAD',
            'UNIDAD', 'GENERO', 'TIPO_INCIDENTE', 'PRIORIDAD_FINAL', 'RECEPCION']
faltan = [a for a in ARCHIVOS if not os.path.exists(f'{DATOS}/{a}')]
assert not faltan, f'Sube a {DATOS} los archivos: {faltan}'

print('✓ Drive montado y librerías importadas')
print(f'   Python {platform.python_version()} · pandas {pd.__version__} · polars {pl.__version__} · '
      f'PySpark {pyspark.__version__}')

In [ ]:
# 1.4 Funciones de monitoreo (notebook 02 del curso: get_memory_info / print_memory_status)
def get_memory_info():
    """Obtiene información sobre el uso de memoria del sistema."""
    memory = psutil.virtual_memory()
    return {'total_gb': round(memory.total / 1024**3, 2), 'disponible_gb': round(memory.available / 1024**3, 2),
            'usado_gb': round(memory.used / 1024**3, 2), 'porcentaje': memory.percent}


def print_memory_status(titulo='Estado de memoria'):
    mem = get_memory_info()
    print(f"\n📊 {titulo}: {mem['usado_gb']} de {mem['total_gb']} GB usados ({mem['porcentaje']:.1f} %), "
          f"{mem['disponible_gb']} GB disponibles")


class MonitorMemoria:
    """Muestrea psutil.virtual_memory().percent cada 50 ms para capturar el PICO de memoria durante una operación."""

    def __init__(self, intervalo=0.05):
        self.intervalo, self.pico, self._activo = intervalo, 0.0, False

    def _muestrear(self):
        while self._activo:
            self.pico = max(self.pico, psutil.virtual_memory().percent)
            time.sleep(self.intervalo)

    def __enter__(self):
        self.pico, self._activo = psutil.virtual_memory().percent, True
        self._hilo = threading.Thread(target=self._muestrear, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._activo = False
        self._hilo.join()


def measure_performance(func, *args, **kwargs):
    """Mide tiempo y memoria de una operación (adaptación de measure_operation, notebook 04 del curso).

    - Tiempo: time.time() antes y después.
    - Memoria: psutil.virtual_memory() (porcentaje del sistema antes y pico durante la operación) y el cambio de
      RSS del proceso Python. El % del sistema incluye la JVM de Spark, que corre en otro proceso.
    """
    proceso = psutil.Process()
    rss_antes = proceso.memory_info().rss / 1024**2
    mem_antes = psutil.virtual_memory().percent
    with MonitorMemoria() as monitor:
        inicio = time.time()
        resultado = func(*args, **kwargs)
        fin = time.time()
    return {'result': resultado, 'time_seconds': fin - inicio, 'mem_pct_antes': mem_antes,
            'mem_pct_pico': monitor.pico, 'memory_delta_mb': proceso.memory_info().rss / 1024**2 - rss_antes}


print('✓ Funciones de medición configuradas')
print_memory_status('Memoria inicial')

## 2. SparkSession optimizada (notebook 02 del curso)

In [ ]:
# Helper del notebook 02 del curso (create_optimized_spark_session), con la zona horaria de Bogotá
def create_optimized_spark_session(app_name='Emergencias123', memory_fraction=0.7):
    """Crea una SparkSession optimizada para Google Colab."""
    num_cores = os.cpu_count()
    driver_memory = f"{max(1, int(get_memory_info()['disponible_gb'] * memory_fraction))}g"   # mínimo 1 GB
    return (SparkSession.builder
            .appName(app_name)
            .master(f'local[{num_cores}]')
            .config('spark.driver.memory', driver_memory)
            .config('spark.driver.maxResultSize', '2g')                           # configuración de la guía
            .config('spark.sql.adaptive.enabled', 'true')
            .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
            .config('spark.sql.adaptive.skewJoin.enabled', 'true')                # configuración de la guía
            .config('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64MB')   # configuración de la guía
            .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
            .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
            .config('spark.sql.session.timeZone', 'America/Bogota')
            .config('spark.ui.showConsoleProgress', 'false')
            .getOrCreate())


m = measure_performance(create_optimized_spark_session)
spark = m['result']
TIEMPO_INICIO_SPARK = m['time_seconds']
print(f"✅ SparkSession creada en {TIEMPO_INICIO_SPARK:.2f} s")
print(f"   - Master: {spark.sparkContext.master}")
print(f"   - Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"   - Driver memory: {spark.conf.get('spark.driver.memory')}")
print(f"   - AQE: {spark.conf.get('spark.sql.adaptive.enabled')} · skewJoin: "
      f"{spark.conf.get('spark.sql.adaptive.skewJoin.enabled')} · Serializador: Kryo · Arrow: activado")
print('\n💡 En Colab, Spark corre en modo local: driver y ejecutores comparten una JVM en un solo nodo (no es un clúster).')

## 3. Diseño del pipeline

Se adapta el `RobustPipeline` del notebook 04 del curso a PySpark, con las etapas del ETL:

1. **Extract:** lectura de los 3 CSV desde Drive con **esquema explícito** (todo texto, nada se infiere) y columnas
   de procedencia (`archivo_origen`, `periodo`).
2. **Transform:** duplicados exactos apartados; `N/A` → nulo (sin imputar); fechas, edad y código de localidad a su
   tipo; textos y códigos estandarizados; **edad homologada a años** según `UNIDAD` (meses, días y horas);
   **minutos entre el inicio del desplazamiento y `RECEPCION`**; marca `observacion_calidad`; **broadcast join** con
   la tabla de códigos.
3. **Validate:** 12 controles de calidad. Tasa de éxito = controles aprobados / controles evaluados × 100.
   Las columnas originales se conservan tal cual; las convertidas van en columnas nuevas (`fecha_inicio`, `edad_num`,
   `edad_anios`, `cod_localidad`, `genero_norm`, …), porque Spark no distingue mayúsculas en los nombres de columna.
4. **Load:** Parquet en Drive particionado por **mes y localidad** (`periodo` / `cod_localidad`), como pide la guía,
   más los agregados y los duplicados apartados.

> **Sobre `RECEPCION`:** el diccionario de datos no define este campo. En los datos siempre es posterior al inicio del
> desplazamiento (mediana de unas 3 horas), así que **no** es el tiempo de respuesta a la llamada. Se reporta con su
> nombre literal (`min_desplazamiento_a_recepcion`) y no se usa como KPI de respuesta.

Cada etapa queda en el log `.txt` con sus registros, errores y tiempo.

In [ ]:
# Normalización común a las tres herramientas: mayúsculas, sin tildes y con espacios simples.
# Necesaria porque los datos traen «USAQUÉN» con tilde y un tipo con doble espacio («AMESUI  - ...»).
CON_TILDE, SIN_TILDE = 'ÁÉÍÓÚÜÑ', 'AEIOUUN'
TABLA_TILDES = str.maketrans(CON_TILDE, SIN_TILDE)
LOCALIDADES = ['USAQUEN', 'CHAPINERO']

ESQUEMA = StructType([StructField(c, StringType(), True) for c in COLUMNAS])   # esquema explícito: nada se infiere
MESES = {'julio': '2026-07', 'agosto': '2026-08', 'septiembre': '2026-09'}
OBSERVACIONES = ['fecha_invalida', 'fecha_fuera_de_mes', 'localidad_sin_dato', 'edad_nula', 'genero_nulo', 'recepcion_nula',
                 'recepcion_antes_de_desplazamiento']
# Edad homologada a años según UNIDAD (valores reales en los datos: Años, Meses, Dias, Horas; «N/A» ya es nulo)
FACTOR_EDAD = {'ANOS': 1.0, 'MESES': 1 / 12, 'DIAS': 1 / 365, 'HORAS': 1 / 8760}


def periodo_de(nombre_archivo):
    """Periodo (mes) del archivo de origen, tomado de su nombre."""
    nombre = nombre_archivo.lower()
    return next(p for mes, p in MESES.items() if mes in nombre)


def evaluar_controles(s):
    """Controles de calidad comunes a PySpark y Pandas a partir de los conteos `s` del pipeline.

    Devuelve la tabla de controles y la tasa de éxito = aprobados / evaluados × 100.
    """
    controles = [
        ('V01', 'Se leyeron los 3 archivos', s['archivos'] == 3, s['archivos']),
        ('V02', 'Columnas requeridas presentes', s['columnas_ok'], s['columnas_ok']),
        ('V03', 'Leídos = aceptados + duplicados exactos apartados', s['leidos'] == s['aceptados'] + s['duplicados'],
         f"{s['leidos']} = {s['aceptados']} + {s['duplicados']}"),
        ('V04', 'Fechas de desplazamiento válidas (0 sin convertir)', s['fechas_invalidas'] == 0, s['fechas_invalidas']),
        ('V05', 'Fechas dentro del mes del archivo (+1 día de tolerancia)', s['fuera_ventana'] == 0, s['fuera_ventana']),
        ('V06', 'Todos los códigos de tipo están en el catálogo', s['sin_catalogo'] == 0, s['sin_catalogo']),
        ('V07', 'El join no cambia el número de filas', s['filas_antes_join'] == s['filas_despues_join'],
         f"{s['filas_antes_join']} → {s['filas_despues_join']}"),
        ('V08', 'Cada localidad tiene un único código', s['localidades_inconsistentes'] == 0, s['localidades_inconsistentes']),
        ('V09', 'Edades dentro de 0–120', s['edades_fuera_rango'] == 0, s['edades_fuera_rango']),
        ('V10', 'Sin duplicados exactos remanentes', s['duplicados_remanentes'] == 0, s['duplicados_remanentes']),
        ('V11', 'El Parquet releído tiene las filas aceptadas', s['parquet_filas'] == s['aceptados'], s['parquet_filas']),
        ('V12', 'RECEPCION nunca es anterior al inicio del desplazamiento', s['recepcion_antes'] == 0, s['recepcion_antes']),
    ]
    tabla = pd.DataFrame(controles, columns=['control', 'descripcion', 'aprobado', 'valor'])
    return tabla, round(tabla['aprobado'].mean() * 100, 1)


class PipelineEmergencias:
    """Pipeline robusto en PySpark (adaptación de RobustPipeline del notebook 04 del curso).

    extract → transform → validate → load, con log en archivo .txt y métricas de cada etapa.
    Regla de limpieza del proyecto: solo se apartan los duplicados EXACTOS (100 % de los atributos iguales) y se
    guardan aparte. Los nulos y demás anomalías se conservan y se marcan en `observacion_calidad`. Nunca se imputa.
    """

    def __init__(self, spark, nombre, destino, log_file):
        self.spark, self.nombre, self.destino, self.log_file = spark, nombre, destino, log_file
        self.metrics = {}
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        open(log_file, 'w', encoding='utf-8').close()

    def log(self, mensaje, nivel='INFO', mostrar=True):
        entrada = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{nivel}] {mensaje}"
        if mostrar:
            print('   ' + entrada)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(entrada + '\n')

    def extract(self, rutas):
        inicio = time.time()
        df = (self.spark.read.csv(rutas, schema=ESQUEMA, sep=';', header=True, encoding='UTF-8')
              .withColumn('archivo_origen', F.col('_metadata.file_name')))
        periodo = F.lit(None).cast('string')
        for mes, p in MESES.items():
            periodo = F.when(F.lower(F.col('archivo_origen')).contains(mes), p).otherwise(periodo)
        df = df.withColumn('periodo', periodo).cache()
        self.metrics['leidos'] = df.count()
        self.metrics['archivos'] = df.select('archivo_origen').distinct().count()
        self.metrics['extract_s'] = time.time() - inicio
        self.log(f"Extract: {self.metrics['leidos']:,} registros de {self.metrics['archivos']} archivos")
        return df

    def transform(self, df, catalogo):
        inicio = time.time()
        # 1) Duplicados exactos (todas las columnas originales iguales): se conserva la primera copia
        ventana = Window.partitionBy(*COLUMNAS).orderBy('archivo_origen')
        df = df.withColumn('_n', F.row_number().over(ventana))
        duplicados = df.filter('_n > 1').drop('_n').cache()
        df = df.filter('_n = 1').drop('_n')
        # 2) Nulos: «N/A» es un valor faltante; se convierte en null (sin imputar)
        for c in ('EDAD', 'UNIDAD', 'GENERO'):
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == 'N/A', None).otherwise(F.col(c)))
        # 3) Tipos y 4) estandarización de textos y códigos
        norm = lambda c: F.trim(F.regexp_replace(F.translate(F.upper(F.col(c)), CON_TILDE, SIN_TILDE), r'\s+', ' '))
        fmt = F.lit('yyyy-MM-dd HH:mm:ss')
        df = (df
              .withColumn('fecha_inicio', F.try_to_timestamp(F.col('FECHA_INICIO_DESPLAZAMIENTO_MOVIL'), fmt))
              .withColumn('fecha_recepcion', F.try_to_timestamp(F.col('RECEPCION'), fmt))
              .withColumn('cod_localidad', F.expr('try_cast(CODIGO_LOCALIDAD AS INT)'))
              .withColumn('edad_num', F.expr('try_cast(EDAD AS INT)'))
              .withColumn('localidad_norm', norm('LOCALIDAD'))
              .withColumn('codigo_tipo', F.trim(F.split(norm('TIPO_INCIDENTE'), ' - ').getItem(0)))
              .withColumn('prioridad', norm('PRIORIDAD_FINAL'))
              .withColumn('genero_norm', norm('GENERO')))
        # 4b) Edad en años según la unidad (meses, días y horas se convierten; una unidad desconocida queda nula)
        factor = F.lit(None).cast('double')
        for unidad, valor in FACTOR_EDAD.items():
            factor = F.when(norm('UNIDAD') == unidad, F.lit(valor)).otherwise(factor)
        # 4c) Minutos entre el inicio del desplazamiento y RECEPCION. No es el «tiempo de respuesta»: en los datos
        #     RECEPCION siempre es posterior al despacho (mediana ~3 h), así que se reporta con su nombre literal.
        df = (df
              .withColumn('edad_anios', F.round(F.col('edad_num') * factor, 2))
              .withColumn('min_desplazamiento_a_recepcion',
                          F.round((F.col('fecha_recepcion').cast('long') - F.col('fecha_inicio').cast('long')) / 60, 1)))
        # 5) Observaciones de calidad: se marcan, no se borran
        inicio_mes = F.to_timestamp(F.concat(F.col('periodo'), F.lit('-01')))
        fin_ventana = F.date_add(F.add_months(inicio_mes, 1), 1)
        condiciones = {
            'fecha_invalida': F.col('fecha_inicio').isNull(),
            'fecha_fuera_de_mes': (F.col('fecha_inicio') < inicio_mes) | (F.col('fecha_inicio') >= fin_ventana),
            'localidad_sin_dato': F.col('cod_localidad').isNull(),
            'edad_nula': F.col('edad_num').isNull(),
            'genero_nulo': F.col('genero_norm').isNull(),
            'recepcion_nula': F.col('fecha_recepcion').isNull(),
            'recepcion_antes_de_desplazamiento': F.col('min_desplazamiento_a_recepcion') < 0}
        obs = F.concat_ws('; ', *[F.when(condiciones[n], F.lit(n)) for n in OBSERVACIONES])
        df = df.withColumn('observacion_calidad', F.when(obs == '', 'OK').otherwise(obs))
        # 6) Enriquecimiento: broadcast join con la tabla de códigos (23 filas)
        self.metrics['filas_antes_join'] = df.count()
        df = df.join(F.broadcast(catalogo), on='codigo_tipo', how='left').cache()
        self.metrics['filas_despues_join'] = df.count()
        self.metrics['transform_s'] = time.time() - inicio
        self.log(f"Transform: {self.metrics['filas_despues_join']:,} aceptados; duplicados exactos apartados: "
                 f"{duplicados.count():,}")
        return df, duplicados

    def validate(self, df, duplicados):
        inicio = time.time()
        s = {k: self.metrics[k] for k in ('archivos', 'leidos', 'filas_antes_join', 'filas_despues_join')}
        s['aceptados'], s['duplicados'] = df.count(), duplicados.count()
        s['columnas_ok'] = all(c in df.columns for c in COLUMNAS)
        fila = df.agg(
            F.sum(F.col('fecha_inicio').isNull().cast('int')).alias('fechas_invalidas'),
            F.sum(F.col('observacion_calidad').contains('fecha_fuera_de_mes').cast('int')).alias('fuera_ventana'),
            F.sum(F.col('descripcion_tipo').isNull().cast('int')).alias('sin_catalogo'),
            F.sum(((F.col('edad_anios') < 0) | (F.col('edad_anios') > 120)).cast('int')).alias('edades_fuera_rango'),
            F.sum((F.col('min_desplazamiento_a_recepcion') < 0).cast('int')).alias('recepcion_antes'),
        ).first().asDict()
        s.update({k: int(v or 0) for k, v in fila.items()})
        s['localidades_inconsistentes'] = (df.groupBy('localidad_norm').agg(F.countDistinct('cod_localidad').alias('n'))
                                           .filter('n > 1').count())
        s['duplicados_remanentes'] = s['aceptados'] - df.select(*COLUMNAS).distinct().count()
        self.stats = s
        self.metrics['validate_s'] = time.time() - inicio
        self.log(f"Validate: controles calculados sobre {s['aceptados']:,} registros")
        return s

    def load(self, df, duplicados):
        inicio = time.time()
        # Particionamiento mes/localidad que pide la guía: periodo (mes del archivo) y código de localidad.
        # repartition por las mismas columnas → un archivo por carpeta (sin archivos sueltos de más).
        (df.repartition('periodo', 'cod_localidad')
           .write.mode('overwrite').partitionBy('periodo', 'cod_localidad').parquet(f'{self.destino}/detalle'))
        duplicados.write.mode('overwrite').parquet(f'{self.destino}/duplicados_eliminados')
        (df.groupBy('periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo').count()
           .write.mode('overwrite').parquet(f'{self.destino}/agregados'))
        self.stats['parquet_filas'] = self.spark.read.parquet(f'{self.destino}/detalle').count()
        self.metrics['load_s'] = time.time() - inicio
        self.log(f"Load: Parquet particionado por periodo/cod_localidad en {self.destino}")

    def run(self, rutas, catalogo):
        self.log('=' * 60)
        self.log(f'INICIANDO PIPELINE: {self.nombre}')
        inicio = time.time()
        try:
            df = self.extract(rutas)
            limpio, duplicados = self.transform(df, catalogo)
            self.validate(limpio, duplicados)
            self.load(limpio, duplicados)
            errores = 0
        except Exception as e:
            self.log(f'Error en el pipeline: {e}', 'ERROR')
            raise
        self.metrics['tiempo_total_s'] = time.time() - inicio
        controles, tasa = evaluar_controles(self.stats)
        self.metrics.update(aceptados=self.stats['aceptados'], duplicados=self.stats['duplicados'],
                            tasa_validaciones_pct=tasa, errores=errores,
                            controles_fallidos=int((~controles['aprobado']).sum()))
        self.log('-' * 60)
        self.log('REPORTE DE VALIDACIÓN')
        for _, c in controles.iterrows():
            self.log(f"{c['control']} {'OK   ' if c['aprobado'] else 'FALLA'} {c['descripcion']} (valor: {c['valor']})",
                     'INFO' if c['aprobado'] else 'WARNING', mostrar=False)
        self.log(f"Registros procesados: {self.metrics['leidos']:,} leídos · {self.metrics['aceptados']:,} aceptados · "
                 f"{self.metrics['duplicados']:,} duplicados exactos apartados")
        self.log(f"Errores de ejecución: {errores} · Tasa de éxito de validaciones: {tasa} % "
                 f"({len(controles) - self.metrics['controles_fallidos']}/{len(controles)} controles)")
        self.log(f"Tiempo de ejecución: {self.metrics['tiempo_total_s']:.2f} s "
                 f"(extract {self.metrics['extract_s']:.2f} · transform {self.metrics['transform_s']:.2f} · "
                 f"validate {self.metrics['validate_s']:.2f} · load {self.metrics['load_s']:.2f})")
        self.spark.catalog.clearCache()
        return controles


def leer_catalogo_spark(spark):
    return (spark.read.csv(f'{RESULTADOS}/catalogo_tipos_derivado.csv', header=True)
            .select('codigo_tipo', 'descripcion_tipo'))


print('✓ PipelineEmergencias definido')

## 4. Ejecución del PoC

In [ ]:
if not os.path.exists(f'{RESULTADOS}/catalogo_tipos_derivado.csv'):
    raise FileNotFoundError('Ejecuta primero el notebook 01: crea la tabla de códigos derivada.')
catalogo_sp = leer_catalogo_spark(spark)
DESTINO = f'{RESULTADOS}/parquet/poc'
LOG = f'{RESULTADOS}/logs/validacion_poc.txt'

pipeline = PipelineEmergencias(spark, 'PoC_Emergencias123', DESTINO, LOG)
with MonitorMemoria() as monitor:
    controles = pipeline.run(RUTAS, catalogo_sp)
print(f"\n✅ PoC completado en {pipeline.metrics['tiempo_total_s']:.2f} s · memoria pico del sistema {monitor.pico:.1f} %")

## 5. Resultados de la validación

In [ ]:
print(f"Tasa de éxito de validaciones: {pipeline.metrics['tasa_validaciones_pct']} %")
print(f"Registros válidos (aceptados / leídos): {pipeline.metrics['aceptados'] / pipeline.metrics['leidos'] * 100:.2f} %")
controles

### 5.1 Log de validación (`resultados/logs/validacion_poc.txt`)

Es el entregable `.txt` que pide la guía: registros procesados, errores y tiempo de ejecución.

In [ ]:
print(open(LOG, encoding='utf-8').read())

## 6. Lectura del Parquet en Drive

Se relee el Parquet desde Drive para comprobar el esquema y las particiones. Gracias al particionamiento por
mes y localidad, un filtro por mes o por localidad solo lee esas carpetas (*partition pruning*).

In [ ]:
detalle = spark.read.parquet(f'{DESTINO}/detalle')
detalle.printSchema()
carpetas = {p: sorted(d for d in os.listdir(f'{DESTINO}/detalle/{p}') if d.startswith('cod_localidad='))
            for p in sorted(os.listdir(f'{DESTINO}/detalle')) if p.startswith('periodo=')}
for p, locs in carpetas.items():
    print(f'{p}: {len(locs)} carpetas de localidad')
print(f'Total: {sum(map(len, carpetas.values()))} particiones mes/localidad')
detalle.groupBy('periodo').count().orderBy('periodo').show()
detalle.select('periodo', 'cod_localidad', 'localidad_norm', 'codigo_tipo', 'fecha_inicio', 'EDAD', 'UNIDAD',
               'edad_anios', 'min_desplazamiento_a_recepcion', 'observacion_calidad').show(5, truncate=False)

### 6.1 Variables derivadas: edad en años y minutos hasta `RECEPCION`

La edad viene en distintas unidades (`Años`, `Meses`, `Dias`, `Horas`); se homologa a años para poder compararla.
La duración entre el inicio del desplazamiento y `RECEPCION` se describe tal como está, sin interpretarla como
tiempo de respuesta.

In [ ]:
detalle.groupBy('UNIDAD').agg(F.count('*').alias('registros'), F.min('edad_num').alias('edad_min'),
                              F.max('edad_num').alias('edad_max'), F.max('edad_anios').alias('max_en_anios')) \
       .orderBy(F.desc('registros')).show()
cuantiles = detalle.approxQuantile('min_desplazamiento_a_recepcion', [0.05, 0.25, 0.5, 0.75, 0.95], 0.001)
con_dato = detalle.filter(F.col('min_desplazamiento_a_recepcion').isNotNull()).count()
print(f'Minutos desplazamiento → RECEPCION ({con_dato:,} registros con ambas fechas):')
print('   p5 / p25 / mediana / p75 / p95 =', ' / '.join(f'{q:.1f}' for q in cuantiles))

## 7. Datos faltantes y anomalías (notebook 03 del curso)

Se usan `analyze_nulls` y `detect_outliers_iqr` del curso. A diferencia del ejemplo del curso, aquí **no** se
eliminan ni se rellenan: la regla del proyecto es conservar los registros y marcarlos en `observacion_calidad`.

In [ ]:
def analyze_nulls(df):
    """Analiza valores nulos en cada columna (notebook 03 del curso)."""
    total_rows = df.count()
    filas = [{'columna': c, 'nulos': df.filter(F.col(c).isNull()).count()} for c in df.columns]
    resultado = pd.DataFrame(filas)
    resultado['porcentaje'] = (resultado['nulos'] / total_rows * 100).round(2)
    return resultado[resultado['nulos'] > 0]


def detect_outliers_iqr(df, column):
    """Detecta outliers con el método IQR (notebook 03 del curso); aquí solo se marcan."""
    q1, q3 = df.filter(F.col(column).isNotNull()).approxQuantile(column, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    marcado = df.withColumn(f'{column}_outlier', (F.col(column) < inferior) | (F.col(column) > superior))
    return marcado, inferior, superior


display(analyze_nulls(detalle.select(*COLUMNAS, 'fecha_inicio', 'edad_num', 'edad_anios', 'cod_localidad',
                                     'genero_norm')))
marcado, inferior, superior = detect_outliers_iqr(detalle, 'edad_anios')
print(f'Edad (años): límites IQR [{inferior:.1f}, {superior:.1f}] → '
      f"{marcado.filter(F.col('edad_anios_outlier')).count()} valores atípicos (se conservan)")
print('\nObservaciones de calidad más frecuentes:')
detalle.groupBy('observacion_calidad').count().orderBy(F.desc('count')).show(8, truncate=False)

## 8. Plan de ejecución y optimizaciones (notebook 03 del curso)

`explain()` muestra cómo ejecuta Spark una consulta. Se verifica la **poda de particiones** y el filtro empujado a la
lectura del Parquet (*predicate pushdown*), y que el join con la tabla de códigos sea un **broadcast join** (la tabla
pequeña se envía completa a cada tarea, sin *shuffle*).

In [ ]:
consulta = (spark.read.parquet(f'{DESTINO}/detalle')
            .filter((F.col('periodo') == '2026-08') & (F.col('cod_localidad') == 1))   # agosto · USAQUÉN
            .groupBy('codigo_tipo').count())
consulta.explain(mode='formatted')
plan_join = (spark.read.csv(RUTAS, schema=ESQUEMA, sep=';', header=True)
             .withColumn('codigo_tipo', F.trim(F.split(F.col('TIPO_INCIDENTE'), ' - ').getItem(0)))
             .join(F.broadcast(catalogo_sp), 'codigo_tipo', 'left'))
plan_join.explain()

## 9. Conclusiones del PoC

- El pipeline ingiere los 3 CSV desde Drive, aplica limpieza y estandarización sin borrar datos (solo aparta los
  duplicados exactos), valida la calidad con 12 controles y deja en Drive el Parquet particionado por mes y localidad
  junto con su log.
- Homologa la edad a años y calcula los minutos entre el despacho y `RECEPCION`, sin presentarlos como tiempo de
  respuesta porque el significado de `RECEPCION` no está documentado.
- Las métricas de rendimiento (3 ejecuciones frente a Pandas) y la escalabilidad están en el notebook 04 (tarea 2.4).

In [ ]:
with open(f'{MEDICIONES}/poc_validacion.json', 'w', encoding='utf-8') as f:
    json.dump({'metricas': pipeline.metrics, 'controles': controles.astype(str).to_dict('records'),
               'mem_pct_pico': monitor.pico}, f, indent=2, ensure_ascii=False, default=str)
spark.stop()
print('✓ Resumen guardado y SparkSession detenida')